# 🚀 Two-Agent News Sentiment Analyzer with Agent-to-Agent Delegation

This notebook implements an agent-to-agent delegation workflow for financial sentiment analysis using AutoGen's **Nested Chats**:
1. Python fetches raw articles and queries the FAISS vector database for calibration examples.
2. The user initiates a chat with the **Senior Sentiment Analyst (CIO) Agent**.
3. The CIO Agent automatically triggers a nested chat, delegating the scoring task to the **Sentiment Scorer Agent**.
4. The Scorer Agent analyzes the articles, calculates scores, and returns them to the CIO Agent.
5. The CIO Agent aggregates the scores, averages the sentiment, and returns the final JSON report back to the user.

## 🛡️ Multi-Layered Safety Guardrails (Loop & Hallucination Prevention)

This workflow is protected by three layers of safety to prevent conversational auto-reply loops and LLM hallucinations under empty prompt inputs:

1. **Orchestration Layer**:
   * **Location**: `User_Proxy` configuration.
   * **Mechanism**: Sets `max_consecutive_auto_reply=0` to ensure that after receiving the CIO's report, the `User_Proxy` immediately halts the conversation instead of sending an empty string to keep it going. It also uses robust termination signature checks (`aggregate_score`, `sentiment_score`).

2. **Backend Handler Layer**:
   * **Location**: `sentiment/functions/tools/custom_reply.py`.
   * **Mechanism**: Implements an input check inside `custom_nested_chat_reply`. If the incoming message is empty or empty-like (`[]`), it returns immediately with a termination message, completely bypassing LLM execution to block invalid runs.

3. **Prompt Guideline Layer**:
   * **Location**: `sentiment/prompts/sentiment_prompt.txt`.
   * **Mechanism**: Enforces a strict LLM guideline under rules instructing the model to return a clean empty list `[]` and `TERMINATE` if it is given a blank or empty articles input, preventing it from hallucinating mock examples.

In [1]:
import sys
import os
from dotenv import load_dotenv

# Ensure the current directory is in the python path for importing modules
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

sentiment_dir = os.path.dirname(notebook_dir)
if sentiment_dir not in sys.path:
    sys.path.insert(0, sentiment_dir)
'''
project_root = os.path.dirname(sentiment_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
'''
# Load environment variables from .env.local
load_dotenv("../.env.local")

True

In [2]:
import json
import datetime
import pandas as pd
import autogen
from finrobot.agents.workflow import FinRobot
from autogen import UserProxyAgent
from functions.aggregator.aggregator import fetch_aggregate_all_news
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Import refactored utility functions
from functions.utils.read_and_clean import read_file_content, extract_and_clean_response
from functions.utils.build import build_vector_store
from functions.tools.prepare_articles import prepare_articles
from functions.agents import create_scorer_agent, create_cio_agent
from functions.tools.custom_reply import custom_nested_chat_reply

# Read and clean environment variables
nvidia_embedding_model = os.getenv("NVIDIA_EMBEDDING_MODEL", "nvidia/nv-embed-v1").strip('"\' ')
nvidia_base_model = os.getenv("NVIDIA_BASE_MODEL", "").strip('"\' ')
nvidia_api_endpoint = os.getenv("NVIDIA_API_ENDPOINT", "https://integrate.api.nvidia.com/v1").strip('"\' ')
nvidia_api_key = os.getenv("NVIDIA_API_KEY", "").strip('"\' ')

print(f"Initializing NVIDIA Embeddings wrapper ({nvidia_embedding_model})...")
embeddings = NVIDIAEmbeddings(
    model=nvidia_embedding_model,
    nvidia_api_key=nvidia_api_key,
    base_url=nvidia_api_endpoint
)

d:\PartnaStudio\sentinel\stack\FinRobot-IntentChain\sentiment\venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")
d:\PartnaStudio\sentinel\stack\FinRobot-IntentChain\sentiment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing NVIDIA Embeddings wrapper (nvidia/nv-embed-v1)...


C:\Users\19178\AppData\Local\Temp\ipykernel_9056\3075562704.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
from functions import llm_config, base_llm_config


HF Model Name: curiousily/Llama-3-8B-Instruct-Finance-RAG:featherless-ai
HF Base URL: https://router.huggingface.co/v1
HF API Key exists: True


In [4]:
ticker = "AAPL"
news_limit = 5  # Score top 5 articles

In [5]:
# Build vector store
db = build_vector_store("../data/financial_sentiment.csv", embeddings, limit_rows=300)

# Instantiate the UserProxyAgent
user_proxy = UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "") and "TERMINATE" in x.get("content", ""),
    max_consecutive_auto_reply=1,
    code_execution_config={"use_docker": False}
)

Loading dataset: ../data/financial_sentiment.csv...
Indexing 300 records into FAISS vector database...
[+] FAISS Local Vector Store created successfully!


In [6]:
# Instantiate scorer and CIO agents
scorer_agent = create_scorer_agent(
    prompt_path="../prompts/sentiment_prompt.txt",
    schema_path="../schema_json/scorer_schema.json",
    llm_config=llm_config
)

cio_agent = create_cio_agent(
    prompt_path="../prompts/cio_prompt.txt",
    schema_path="../schema_json/sentiment_schema.json",
    output_schema_path="../schema_json/cio_output_schema.json",
    scored_articles_path="../schema_json/cio_scored_articles.json",
    llm_config=base_llm_config
)

In [ ]:
# Prepare the news articles
print(f"Fetching consolidated news feed for {ticker}...")
df_news = fetch_aggregate_all_news(symbol=ticker, limit=100)

if df_news.empty:
    raise ValueError(f"No news articles found for symbol {ticker}.")
    
articles_to_analyze = prepare_articles(df_news, db, limit=news_limit)

# Modular test override: Define Option B batching custom reply function inline in the notebook
def notebook_extract_json_array(text):
    """Extracts a JSON list or dict from a string, supporting markdown wraps."""
    if not text:
        return None
    text_stripped = text.strip()
    try:
        return json.loads(text_stripped)
    except Exception:
        pass
    
    start_list = text.find('[')
    end_list = text.rfind(']')
    start_dict = text.find('{')
    end_dict = text.rfind('}')
    
    start = -1
    end = -1
    if start_list != -1 and end_list != -1:
        start = start_list
        end = end_list
    if start_dict != -1 and end_dict != -1:
        if start == -1 or start_dict < start:
            start = start_dict
            end = end_dict
            
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
            
    return None

def notebook_merge_scored_results(batch_results):
    """Merges multiple scored batch responses from the Scorer Agent."""
    merged_map = {}
    for result in batch_results:
        items = []
        if isinstance(result, dict):
            items = [result]
        elif isinstance(result, list):
            items = result
        else:
            continue
            
        for item in items:
            if not isinstance(item, dict):
                continue
            t = item.get("ticker")
            if not t:
                continue
            
            articles = item.get("articles", [])
            if not isinstance(articles, list):
                articles = []
                
            if t not in merged_map:
                merged_map[t] = {
    "ticker": t,
    "metadata": {
        "timestamp": item.get("metadata", {}).get("timestamp", ""),
        "article_count": 0
    },
    "articles": []
}
            
            merged_map[t]["articles"].extend(articles)
            
    for t, entry in merged_map.items():
        entry["metadata"]["article_count"] = len(entry["articles"])
        
    if len(merged_map) == 1:
        return list(merged_map.values())[0]
    return list(merged_map.values())

def notebook_custom_nested_chat_reply(chat_queue, recipient, messages, sender, config):
    """Custom reply handler with transparent batching (cycles of 5) for modular testing."""
    from autogen import initiate_chats
    
    last_msg = messages[-1].get("content", "").strip() if messages else ""
    if not last_msg or last_msg == "[]":
        return True, "No articles provided to score. TERMINATE"

    articles_list = notebook_extract_json_array(last_msg)

    if not articles_list or not isinstance(articles_list, list):
        chats_to_run = recipient._get_chats_to_run(chat_queue, recipient, messages, sender, config)
        if not chats_to_run:
            return True, None
        res = initiate_chats(chats_to_run)
        scorer_summary = res[-1].summary
    else:
        batch_size = 5
        batch_results = []
        
        for i in range(0, len(articles_list), batch_size):
            batch = articles_list[i:i + batch_size]
            print(f"[*] [Notebook Batcher] Scoring articles {i+1} to {min(i+batch_size, len(articles_list))} of {len(articles_list)}")
            
            temp_chat_queue = []
            for chat_config in chat_queue:
                temp_config = chat_config.copy()
                temp_config["message"] = (
                    "Please score the following articles according to your instructions:\n\n"
                    f"{json.dumps(batch, indent=2)}\n\n"
                    "Respond with the list of scored articles."
                )
                temp_chat_queue.append(temp_config)
                
            chats_to_run = recipient._get_chats_to_run(temp_chat_queue, recipient, messages, sender, config)
            res = initiate_chats(chats_to_run)
            
            summary_content = res[-1].summary
            scored_data = notebook_extract_json_array(summary_content)
            if scored_data is not None:
                batch_results.append(scored_data)
            else:
                try:
                    clean_content = summary_content
                    if clean_content.endswith("TERMINATE"):
                        clean_content = clean_content[:-9].strip()
                    batch_results.append(json.loads(clean_content))
                except Exception:
                    pass
                    
        merged_result = notebook_merge_scored_results(batch_results)
        scorer_summary = json.dumps(merged_result, indent=2)
    
    recipient.send(
        message=scorer_summary,
        recipient=sender,
        request_reply=False,
        silent=True
    )
    
    return False, None

# Configure the nested chat on the CIO Agent.
# When the User Proxy sends raw articles to the CIO, the CIO delegates them to the Scorer.
nested_chats = [
    {
        "recipient": scorer_agent,
        "message": lambda recipient, messages, sender, config: (
            "Please score the following articles according to your instructions:\n\n"
            f"{messages[-1]['content']}\n\n"
            "Respond with the list of scored articles."
        ),
        "summary_method": "last_msg",
        "max_turns": 1,
    }
]

cio_agent.register_nested_chats(
    nested_chats,
    trigger=user_proxy,
    reply_func_from_nested_chats=notebook_custom_nested_chat_reply
)

# Step 1 & 2: Initiate chat directly with the CIO agent.
# The user proxy passes the raw articles.
print("\n[+] Triggering Agent-to-Agent Delegation Chat...")
user_proxy.initiate_chat(
    cio_agent,
    message=f"Analyze the following scored articles for ticker {ticker}:\n\n" + json.dumps(articles_to_analyze, indent=2)
)

final_report_msg = extract_and_clean_response(user_proxy, cio_agent, is_json=True)

print("\n================ FINAL REPORT ================")
try:
    final_report = json.loads(final_report_msg)
    print(json.dumps(final_report, indent=2))
except Exception as e:
    print(f"Error parsing final report: {e}")
    print("Raw Output:")
    print(final_report_msg)


Fetching consolidated news feed for AAPL...
[+] Fetching OpenBB YFINANCE...
[+] Fetching OpenBB TMX...
    -> OpenBB TMX Error: Results not found.
[+] Fetching OpenBB FMP...
    -> OpenBB FMP Error: 
[Error] -> Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/
[+] Fetching OpenBB TIINGO...
    -> OpenBB TIINGO Error: 
[Error] -> Unauthorized Tiingo request -> You do not have permission to access the News API
[+] Fetching OpenBB BIZTOC...
[+] Fetching OpenBB BENZINGA...
[+] Fetching Custom ALPHA-VANTAGE...
[+] Fetching Custom NEWS_API...
[+] Fetching Custom SEEKING-ALPHA...
Seeking Alpha Error: 403 Client Error: Forbidden for url: https://seeking-alpha-api.p.rapidapi.com/news/v2/list-by-symbol?symbol=AAPL&size=100
[+] Fetching Custom NASDAQ...
[+] Fetching Custom FINVIZ...

[+] Triggering Agent-to-Agent Delegation Chat...
User

In [8]:
print(final_report_msg)

{
  "ticker": "AAPL",
  "metadata": {
    "timestamp": "2026-06-16T23:14:00.000Z",
    "article_count": 2
  },
  "articles": [
    {
      "title": "Apple's OLED MacBook push raises stakes for BOE and Samsung display race",
      "source": "Finviz Source",
      "published_at": "2026-06-16 23:14:00",
      "sentiment_label": "Positive",
      "sentiment_score": 0.83,
      "confidence": 0.92,
      "risk_factors": ["regulatory_headwind", "supply_chain_disruption"],
      "reasoning_summary": "Apple's OLED MacBook push raises stakes for BOE and Samsung display race",
      "flagged": false,
      "flag_reason": null
    },
    {
      "title": "Apple moves private cloud compute to third party with Google Cloud AI",
      "source": "Finviz Source",
      "published_at": "2026-06-16 22:55:00",
      "sentiment_label": "Positive",
      "sentiment_score": 0.92,
      "confidence": 0.95,
      "risk_factors": ["regulatory_headwind", "supply_chain_disruption"],
      "reasoning_summary": "Ap